# Compare corrected v2 runs
CPU only. This notebook excludes preliminary v1 runs, checks that core paired cells exist, and reports retention, learning, replay behavior, and cost with both normalized and summed metrics. It also compares MFR at 10% replay with the higher-budget random control when those runs are present.

In [ ]:
import os, sys
import pandas as pd
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/mfr-dpo'
    DRIVE_DIR = '/content/drive/MyDrive/CSCI544/mfr-dpo'
else:
    REPO = '..'
    DRIVE_DIR = os.environ.get('MFR_DRIVE_DIR', '../drive')
sys.path.insert(0, f'{REPO}/src')
import mfr_analysis
runs = mfr_analysis.load_runs(DRIVE_DIR, include_preliminary=False, completed_only=True)


In [ ]:
mfr_analysis.validate_compatible_runs(runs, strict=False)
display(runs[['run_name','order_id','method','seed','stage']].drop_duplicates().sort_values(['order_id','seed','method','stage']))


In [ ]:
print('PRIMARY: length-normalized relative accuracy')
display(mfr_analysis.summary_table(runs, score='accuracy').round(2))
display(mfr_analysis.retention_table(runs, score='accuracy').round(2))
display(mfr_analysis.new_learning_table(runs, score='accuracy').round(2))


In [ ]:
print('SECONDARY: summed DPO relative accuracy')
display(mfr_analysis.summary_table(runs, score='accuracy_sum').round(2))
print('PAIR-LEVEL MFR MINUS RANDOM RETENTION CHANGE, WITH 95% BOOTSTRAP INTERVALS')
display(mfr_analysis.paired_retention_comparison(runs, margin='margin').round(2))
display(mfr_analysis.paired_retention_comparison(runs, margin='margin_sum').round(2))
print('PAIR-LEVEL MFR (10%) MINUS RANDOM_HIGH (14.3%) RETENTION CHANGE')
display(mfr_analysis.paired_retention_comparison(runs, baseline='random_high', margin='margin').round(2))
display(mfr_analysis.paired_retention_comparison(runs, baseline='random_high', margin='margin_sum').round(2))
print('COST AND REPLAY AUDIT')
display(mfr_analysis.cost_table(runs).round(2))
display(mfr_analysis.replay_summary(runs))


Interpret MFR as successful only if it improves paired retention over random replay across both orders/seeds without reducing final new-stage validation accuracy by more than 2 points. With only one seed or one order, report the result as preliminary rather than a conclusion.